# NeoOLAF × RAGTree — Unified 4-Dataset Native Benchmark v1.7

**MAVEN-ERE relation correction on top of v1.5.**

v1.4 solved event extraction on the fixed development document (21/21 event clusters and 12/12 relation-endpoint clusters reached by Layer 1). v1.5 then over-corrected relation precision: its skeptic cascade reduced the graph to a few edges but removed every true relation. v1.7 freezes/reuses the v1.4 event inventory and changes only MAVEN Layer 2: no relation-time coreference merging, three complementary high-recall global proposals, a single PRECONDITION-aware calibrated adjudicator, and post-L12 per-stage gold recall diagnostics. EventStoryLine, FinCausal and CausalBank remain frozen. No file under `src/neoolaf` is modified.


In [1]:
from pathlib import Path
import os, sys, json, time, traceback
from pprint import pprint

def find_project_root():
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError("NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"
for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate
import ragtree_dataset_adapters_v1_7 as adapters
import eventstoryline_native_ablation_v1_7 as esl_v17

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Adapter v1.7 self-test:")
pprint(adapters.offline_self_test())

c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
Adapter v1.7 self-test:
{'cache_root': 'C:\\Users\\GALENC~1\\AppData\\Local\\Temp\\neoolaf_ragtree_unified4_v1_1',
 'causalbank': 'deterministic all ordered non-self pairs; no pair-level LLM '
               'pruning',
 'causalbank_hash_a': 'EVENT_0cc175b9c0f1b6a8',
 'causalbank_relations': ['BECAUSE', 'THEREFORE'],
 'datasets': ['fincausal', 'maven_ere', 'causalbank'],
 'fincausal': 'deterministic boundary union + role-hint pair fallback',
 'maven_ere': 'frozen/reused v1.4 event inventory + singleton anchors + three '
              'high-recall graph proposals + PRECONDITION-aware adjudication + '
              'posthoc stage recall diagnostics',
 'ok': True,
 'patch_version': 'unified4-v1.7'}


## Run controls — safe defaults

In [2]:
# PAID EXECUTION GUARD
RUN_PAID = True              # Preflight first; v1.7 reuses frozen v1.4 Layer 1 when available.
RUN_MODE = "one_doc"           # one_doc | smoke5 | full
RUN_DATASETS = ["maven_ere"]  # v1.7 is MAVEN relation-only; ESL/FinCausal/CausalBank stay frozen.

MODEL_NAME = "openai/gpt-oss-20b"
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
REASONING_EFFORT = "minimal"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 180

DOCUMENT_WORKERS = 1
LAYER_WORKERS = 4
VERBOSE = True

FORCE_RUN = {
    "eventstoryline": False,
    "fincausal": False,
    "maven_ere": False,
    "causalbank": False,
}

# The three new datasets use the FIRST FIVE normalized records, which are the fixed
# five-document files inspected during adapter design. Once selected, record keys
# are persisted and cannot silently change.
SMOKE_DOCUMENT_IDS = {
    "eventstoryline": [
        "EventStoryLine - 1_10ecbplus",
        "EventStoryLine - 1_11ecbplus",
        "EventStoryLine - 1_12ecbplus",
        "EventStoryLine - 1_13ecbplus",
        "EventStoryLine - 1_14ecbplus",
    ],
    "fincausal": [],
    "maven_ere": [],
    "causalbank": [],
}

# ONE-DOC is only a sanity gate; it is not the final benchmark threshold.
ONE_DOC_SANITY = {
    "eventstoryline": {"relation_f1": 0.15, "endpoint_recall": 0.50},
    "fincausal":      {"relation_f1": 0.50, "endpoint_recall": 0.50},
    "maven_ere":      {"relation_f1": 0.05, "endpoint_recall": 0.40},
    "causalbank":     {"relation_f1": 0.20, "endpoint_recall": 0.60},
}

# After the ONE AND ONLY paid smoke-5, these are the practical stopping bands.
SMOKE_ACCEPT_MIN_F1 = {
    "eventstoryline": 0.15,
    "fincausal": 0.75,
    "maven_ere": 0.08,
    "causalbank": 0.20,
}
SMOKE_TARGET_F1 = {
    "eventstoryline": 0.20,
    "fincausal": 0.90,
    "maven_ere": 0.15,
    "causalbank": 0.35,
}

assert RUN_MODE in {"one_doc", "smoke5", "full"}
assert all(k in expstate.DATASET_KEYS for k in RUN_DATASETS)
print("RUN_PAID =", RUN_PAID, "| RUN_MODE =", RUN_MODE, "| datasets =", RUN_DATASETS)

RUN_PAID = True | RUN_MODE = one_doc | datasets = ['maven_ere']


## Zero-cost path + ontology + dataset preflight

In [3]:
RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)
RAW_ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)
DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)

# NeoOLAF's seed loader consumes TBox classes/properties. Keep original OWL-Time
# and FIBO; use the compact, already-validated schema views for EventKG/WordNet.
COMPAT_DIR = EXPERIMENT_ROOT / "ontology_compat"
COMPAT_ONTOLOGIES = {
    "maven_ere": COMPAT_DIR / "EventKGSchema_NeoOLAF.ttl",
    "causalbank": COMPAT_DIR / "wordnet_neoolaf_seed.ttl",
}
for k,p in COMPAT_ONTOLOGIES.items():
    assert p.exists(), (k, p)

ONTOLOGY_FILES = dict(RAW_ONTOLOGY_FILES)
ONTOLOGY_FILES["maven_ere"] = COMPAT_ONTOLOGIES["maven_ere"]
ONTOLOGY_FILES["causalbank"] = COMPAT_ONTOLOGIES["causalbank"]

print("RAGTREE_ROOT      :", RAGTREE_ROOT)
print("PREPROCESSED_DIR  :", PREPROCESSED_DIR)
print("ONTOLOGY_ROOT     :", ONTOLOGY_ROOT)
print("\\nEffective NeoOLAF seeds:")
for k,v in ONTOLOGY_FILES.items():
    print(f"  {k:15s} -> {v}")
print("\\nNormalized JSONLs:")
for k,v in DATASET_FILES.items():
    print(f"  {k:15s} -> {v}")

from neoolaf.ontology.loader import SeedOntologyLoader
def seed_counts(path):
    seed = SeedOntologyLoader().load(str(path))
    return {"classes": len(seed.classes_by_uri), "properties": len(seed.properties_by_uri)}

SEED_COUNTS = {}
for k in expstate.DATASET_KEYS:
    counts = seed_counts(ONTOLOGY_FILES[k])
    SEED_COUNTS[k] = counts
    print(f"{k:15s} seed counts -> {counts}")
    if not counts["classes"] and not counts["properties"]:
        raise RuntimeError(f"{k}: ontology exposes 0 classes/properties: {ONTOLOGY_FILES[k]}")

assert SEED_COUNTS["fincausal"]["classes"] >= 1000 and SEED_COUNTS["fincausal"]["properties"] >= 500
assert SEED_COUNTS["maven_ere"]["classes"] >= 1 and SEED_COUNTS["maven_ere"]["properties"] >= 3
assert SEED_COUNTS["causalbank"]["classes"] >= 18 and SEED_COUNTS["causalbank"]["properties"] >= 32

print("\\nONTOLOGY PREFLIGHT: OK")
print("No paid/API call has been made.")

RAGTREE_ROOT      : C:\Users\galencarmedeiro\RAGTree
PREPROCESSED_DIR  : C:\Users\galencarmedeiro\RAGTree\preprocessed
ONTOLOGY_ROOT     : C:\Users\galencarmedeiro\RAGTree\data\ontology
\nEffective NeoOLAF seeds:
  eventstoryline  -> C:\Users\galencarmedeiro\RAGTree\data\ontology\OWLTime\time.ttl
  fincausal       -> C:\Users\galencarmedeiro\RAGTree\data\ontology\FIBO-CorePlus\fibo-core-plus.ttl
  maven_ere       -> C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\ontology_compat\EventKGSchema_NeoOLAF.ttl
  causalbank      -> C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\ontology_compat\wordnet_neoolaf_seed.ttl
\nNormalized JSONLs:
  eventstoryline  -> C:\Users\galencarmedeiro\RAGTree\preprocessed\eventstoryline.jsonl
  fincausal       -> C:\Users\galencarmedeiro\RAGTree\preprocessed\fincausal.jsonl
  maven_ere       -> C:\Users\galencarmedeiro\RAGTree\preprocessed\maven_ere.jsonl
  causalbank      -> C:\Users\galencarmedeiro\RAGTree\preprocessed\causalbank.jsonl
e

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#dateTime, Converter=<built-in method fromisoformat of type object at 0x00007FF88CAC7970>
Traceback (most recent call last):
  File "c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
           ^^^^^^^^^^^^^^^^^^
ValueError: Invalid isoformat string: '2025-10-6T18:00:00'


fincausal       seed counts -> {'classes': 1009, 'properties': 509}
maven_ere       seed counts -> {'classes': 1, 'properties': 3}
causalbank      seed counts -> {'classes': 18, 'properties': 32}
\nONTOLOGY PREFLIGHT: OK
No paid/API call has been made.


## Offline dataset audit + anti-leak check

In [4]:
dataset_rows = {k: expstate.read_jsonl(v) for k,v in DATASET_FILES.items()}

def audit_dataset(k, rows):
    rel_counts = {}
    entity_counts = []
    for r in rows:
        entity_counts.append(len(r.get("entities") or {}))
        for rel,pairs in (r.get("relations") or {}).items():
            rel_counts[rel] = rel_counts.get(rel, 0) + len(pairs or [])
    return {
        "dataset": k,
        "records": len(rows),
        "relations": rel_counts,
        "mean_gold_entities": (sum(entity_counts)/len(entity_counts)) if entity_counts else 0,
        "has_ontology_links": sum("ontology_links" in r for r in rows),
    }

audit = [audit_dataset(k, dataset_rows[k]) for k in expstate.DATASET_KEYS]
for row in audit:
    pprint(row)

expected_relation_vocab = {
    "eventstoryline": {"PRECONDITION", "FALLING_ACTION"},
    "fincausal": {"CAUSE"},
    "maven_ere": {"CAUSE", "PRECONDITION"},
    "causalbank": {"BECAUSE", "THEREFORE"},
}
for row in audit:
    observed = {str(x).upper() for x in row["relations"] if str(x).lower() not in {"null","none",""}}
    assert observed == expected_relation_vocab[row["dataset"]], (row["dataset"], observed)

print("\nPipeline-visible sanitization:")
for k in expstate.DATASET_KEYS:
    sample = expstate.strip_gold(dataset_rows[k][0])
    forbidden = {"entities","relations","pred_relations","ontology_links"} & set(sample)
    assert not forbidden, (k, forbidden)
    print(k, "OK; visible keys =", sorted(sample.keys()))

print("\nDATASET / ANTI-LEAK PREFLIGHT: OK")

{'dataset': 'eventstoryline',
 'has_ontology_links': 0,
 'mean_gold_entities': 11.753950338600452,
 'records': 443,
 'relations': {'FALLING_ACTION': 4880, 'PRECONDITION': 4760, 'null': 55}}
{'dataset': 'fincausal',
 'has_ontology_links': 0,
 'mean_gold_entities': 1.9493278179937952,
 'records': 967,
 'relations': {'CAUSE': 929}}
{'dataset': 'maven_ere',
 'has_ontology_links': 3516,
 'mean_gold_entities': 23.6754835039818,
 'records': 3516,
 'relations': {'CAUSE': 8420, 'PRECONDITION': 37594}}
{'dataset': 'causalbank',
 'has_ontology_links': 0,
 'mean_gold_entities': 14.096296296296297,
 'records': 1080,
 'relations': {'BECAUSE': 43784, 'THEREFORE': 123326}}

Pipeline-visible sanitization:
eventstoryline OK; visible keys = ['document_id', 'sentences', 'text', 'title', 'tokens', 'type']
fincausal OK; visible keys = ['document_id', 'sentences', 'text', 'title', 'tokens', 'type']
maven_ere OK; visible keys = ['document_id', 'sentences', 'text', 'title', 'tokens', 'type']
causalbank OK; vis

## Dataset-specific v1.3 configs

In [5]:
CONFIGS = {
    "eventstoryline": {
        "profile": EXPERIMENT_ROOT / "configs/eventstoryline_profile_native_ablation_v1_7.json",
        "guidance": EXPERIMENT_ROOT / "configs/guidance_eventstoryline_native_ablation_v1_7.json",
        "task": EXPERIMENT_ROOT / "configs/eventstoryline_task_guidance_v1_7.json",
        "catalog": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_aliases.json",
        "version": "v1.7",
    },
    "fincausal": {
        "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1_3.json",
        "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1_3.json",
        "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1_3.json",
        "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
        "version": "unified-v1.3.1-selection-hotfix",
    },
    "maven_ere": {
        "profile": EXPERIMENT_ROOT / "configs/maven_ere_profile_unified_v1_7.json",
        "guidance": EXPERIMENT_ROOT / "configs/maven_ere_guidance_unified_v1_7.json",
        "task": EXPERIMENT_ROOT / "configs/maven_ere_task_guidance_unified_v1_7.json",
        "catalog": EXPERIMENT_ROOT / "ontology/maven_ere_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/maven_ere_relation_aliases.json",
        "version": "unified-v1.7",
    },
    "causalbank": {
        "profile": EXPERIMENT_ROOT / "configs/causalbank_profile_unified_v1_3.json",
        "guidance": EXPERIMENT_ROOT / "configs/causalbank_guidance_unified_v1_3.json",
        "task": EXPERIMENT_ROOT / "configs/causalbank_task_guidance_unified_v1_3.json",
        "catalog": EXPERIMENT_ROOT / "ontology/causalbank_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/causalbank_relation_aliases.json",
        "version": "unified-v1.3",
    },
}
for k,cfg in CONFIGS.items():
    for name,p in cfg.items():
        if name != "version":
            assert Path(p).exists(), (k,name,p)
print("CONFIG PREFLIGHT: OK")

CONFIG PREFLIGHT: OK


## Metric-aware persistent budget/state guard + FinCausal dev-record validity

The old empty FinCausal first row is not accepted as the development sanity document.

Selection rule is deterministic and fixed:

> **first source-order FinCausal record containing at least one scored `CAUSE` relation**

This is only a validity criterion for the one-document development sanity gate. It is not performance-based selection and it does not alter the future smoke-5 selection.

The selected gold counts are inspected by the notebook controller only. `strip_gold(...)` still removes `entities`, `relations`, `pred_relations`, and `ontology_links` before NeoOLAF runs.


In [6]:
STATE_DIR = EXPERIMENT_ROOT / "state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
TEMPLATE_MANIFEST = STATE_DIR / "development_manifest_TEMPLATE_v1.json"
LIVE_MANIFEST = STATE_DIR / "development_manifest_v1.json"
manifest = expstate.load_manifest(LIVE_MANIFEST, TEMPLATE_MANIFEST)

KNOWN_RUN_ROOTS = [
    EXPERIMENT_ROOT / "runs" / "unified4_v1",
    EXPERIMENT_ROOT / "runs" / "unified4_v1_1",
    EXPERIMENT_ROOT / "runs" / "unified4_v1_2",
    EXPERIMENT_ROOT / "runs" / "unified4_v1_3",
    EXPERIMENT_ROOT / "runs" / "unified4_v1_3_1",
    EXPERIMENT_ROOT / "runs" / "unified4_v1_4",
    EXPERIMENT_ROOT / "runs" / "unified4_v1_5",
    EXPERIMENT_ROOT / "runs" / "unified4_v1_7",
]

def metric_value(m, key, default=0.0):
    try:
        return float((m or {}).get(key, default) or 0.0)
    except Exception:
        return default

def one_doc_sanity(dataset_key, result):
    if dataset_key == "eventstoryline":
        # The ESL v1.7 development run was already explicitly accepted before this unified patch.
        return {"passed": True, "reason": "previously accepted v1.7 development result"}
    rel = result.get("relation_metrics") or result.get("projected_relation_evaluation") or {}
    ep = result.get("endpoint_metrics") or result.get("relation_endpoint_evaluation") or {}
    th = ONE_DOC_SANITY[dataset_key]
    rf1 = metric_value(rel, "f1")
    er = metric_value(ep, "recall")
    passed = rf1 >= th["relation_f1"] and er >= th["endpoint_recall"]
    return {
        "passed": passed,
        "relation_f1": rf1,
        "endpoint_recall": er,
        "required_relation_f1": th["relation_f1"],
        "required_endpoint_recall": th["endpoint_recall"],
    }

def newest_existing_one_doc_result(dataset_key):
    found = []
    for root in KNOWN_RUN_ROOTS:
        d = root / dataset_key / "one_doc"
        if d.exists():
            for p in d.glob("*/posthoc_evaluation.json"):
                try:
                    found.append((p.stat().st_mtime, p))
                except OSError:
                    pass
    if not found:
        return None, None
    _, path = sorted(found, key=lambda x: x[0])[-1]
    try:
        return adapters.read_json(path), path
    except Exception:
        return None, path

recovered_results = {}
state_report = []

# ESL stays accepted exactly as before.
manifest["eventstoryline"]["one_doc_completed"] = True
manifest["eventstoryline"]["status"] = "READY_5"
manifest["eventstoryline"]["best_version"] = "v1.7"

for dataset_key in ["fincausal", "maven_ere", "causalbank"]:
    result, path = newest_existing_one_doc_result(dataset_key)
    if result is not None:
        recovered_results[dataset_key] = result
        sanity = one_doc_sanity(dataset_key, result)
        manifest[dataset_key]["previous_one_doc_evaluation"] = str(path)
        manifest[dataset_key]["previous_one_doc_sanity"] = sanity
        if sanity["passed"]:
            manifest[dataset_key]["one_doc_completed"] = True
            manifest[dataset_key]["status"] = "READY_5"
        else:
            manifest[dataset_key]["one_doc_completed"] = False
            manifest[dataset_key]["status"] = "NEEDS_FIX"
    else:
        # Do NOT trust an old boolean that may have been produced by the pre-v1.3 manifest bug.
        if dataset_key in {"fincausal", "maven_ere"}:
            manifest[dataset_key]["one_doc_completed"] = False
            manifest[dataset_key]["status"] = "NEEDS_FIX"
            manifest[dataset_key]["previous_one_doc_sanity"] = {
                "passed": False,
                "reason": "No recoverable posthoc_evaluation.json; old READY_5 boolean is not sufficient."
            }

# v1.3.1: freeze an actually evaluable FinCausal development record BEFORE any paid call.
# This uses gold only as a controller-side validity check; the pipeline-visible copy is
# still produced later via strip_gold(...).
fc_dev = expstate.select_records_for_mode(
    "fincausal", dataset_rows["fincausal"], manifest, "one_doc",
    preferred_document_ids=None,
)[0]
fc_dev_contract = expstate.gold_contract_summary("fincausal", fc_dev)
manifest["fincausal"]["v1_3_1_dev_gold_contract"] = fc_dev_contract
print("\nFinCausal v1.3.1 frozen development record (ZERO-COST controller check):")
pprint(fc_dev_contract)
assert fc_dev_contract["gold_target_relation_count"] > 0, (
    "Refusing a paid FinCausal one-doc run: selected record has no scored CAUSE relation."
)
assert fc_dev_contract["gold_entity_count"] >= 2, (
    "Refusing a paid FinCausal one-doc run: selected record does not expose the expected gold endpoint inventory."
)


# MAVEN v1.4 zero-cost source-candidate reachability audit. The candidate pool is
# generated from visible tokens first; gold is consulted only afterward for this controller diagnostic.
maven_dev = expstate.select_records_for_mode(
    "maven_ere", dataset_rows["maven_ere"], manifest, "one_doc", preferred_document_ids=None,
)[0]
maven_preview = adapters.offline_maven_rescue_candidate_preview(maven_dev)
manifest["maven_ere"]["v1_7_frozen_layer1_source_rescue_preview"] = maven_preview
print("\nMAVEN frozen-v1.4 ZERO-COST source rescue ceiling (unchanged in v1.5):")
pprint(maven_preview)
assert maven_preview["relation_endpoint_cluster_recall_ceiling"] >= 0.80, (
    "MAVEN source-only rescue candidate pool does not cover enough relation endpoints; refusing paid run."
)

# CausalBank v1.3 is a deterministic relaxation of the already successful endpoint inventory.
# Preview is source-derived first; gold is consulted only afterward to measure the exact rule.
cb_preview = adapters.offline_causalbank_dense_projection_preview(dataset_rows["causalbank"][0])
print("\nCausalBank v1.3 ZERO-COST dense projection preview:")
pprint(cb_preview)
if cb_preview.get("available"):
    cb_sanity = one_doc_sanity("causalbank", cb_preview)
    manifest["causalbank"]["v1_3_offline_dense_preview"] = cb_preview
    manifest["causalbank"]["v1_3_offline_dense_sanity"] = cb_sanity
    if cb_sanity["passed"]:
        manifest["causalbank"]["one_doc_completed"] = True
        manifest["causalbank"]["status"] = "READY_5"
        manifest["causalbank"]["best_version"] = "unified-v1.3"
        recovered_results["causalbank_v1_3_preview"] = cb_preview

expstate.save_manifest(LIVE_MANIFEST, manifest)

for k in expstate.DATASET_KEYS:
    prev = manifest[k].get("previous_one_doc_sanity") or manifest[k].get("v1_3_offline_dense_sanity") or {}
    state_report.append({
        "dataset": k,
        "status": manifest[k].get("status"),
        "one_doc_completed": manifest[k].get("one_doc_completed"),
        "best_version": manifest[k].get("best_version"),
        "previous_relation_F1": prev.get("relation_f1"),
        "previous_endpoint_R": prev.get("endpoint_recall"),
        "reason": prev.get("reason"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(state_report))
except Exception:
    pprint(state_report)

print("\nPaid one-doc work still required:")
for k in RUN_DATASETS:
    if not manifest[k].get("one_doc_completed"):
        print(" -", k)
print("\nLive manifest:", LIVE_MANIFEST)


FinCausal v1.3.1 frozen development record (ZERO-COST controller check):
{'dataset': 'fincausal',
 'document_id': 'FinCausal - a39155e8dec741b9',
 'gold_entity_count': 2,
 'gold_target_relation_count': 1,
 'line_index': 10,
 'record_key': 'fincausal:10:7551d279863a4a5c',
 'title': '0001.00005.1'}

MAVEN frozen-v1.4 ZERO-COST source rescue ceiling (unchanged in v1.5):
{'all_gold_cluster_recall_ceiling': 1.0,
 'all_gold_event_clusters': 21,
 'all_gold_event_clusters_reachable': 21,
 'gold_used_only_after_candidate_generation': True,
 'relation_endpoint_cluster_recall_ceiling': 1.0,
 'relation_endpoint_clusters_reachable': 12,
 'relation_endpoint_gold_clusters': 12,
 'source_candidate_count': 123}

CausalBank v1.3 ZERO-COST dense projection preview:
{'available': True,
 'endpoint_metrics': {'f1': 1.0,
                      'gold_unique': 11,
                      'precision': 1.0,
                      'pred_mapped_unique': 11,
                      'recall': 1.0,
                      '

,dataset,status,one_doc_completed,best_version,previous_relation_F1,previous_endpoint_R,reason
0,eventstoryline,READY_5,True,v1.7,NaN,NaN,None
1,fincausal,READY_5,True,unified-v1.3.1-selection-hotfix,1.000000,1.000000,None
2,maven_ere,NEEDS_FIX,False,unified-v1.6,0.000000,0.333333,None
3,causalbank,READY_5,True,unified-v1.3,0.254237,0.727273,None



Paid one-doc work still required:
 - maven_ere

Live manifest: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\state\development_manifest_v1.json


## Native Layer 0–12 runner — gold written only after Layer 12

In [7]:
RUNS_ROOT = EXPERIMENT_ROOT / "runs" / "unified4_v1_7"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def safe_dir_name(text):
    import re
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

def run_one_record(dataset_key, gold_record):
    cfg = CONFIGS[dataset_key]
    rkey = expstate.record_key(dataset_key, gold_record)

    # Controller-only integrity contract. Never passed to NeoOLAF.
    pre_gold_contract = expstate.gold_contract_summary(dataset_key, gold_record)
    if dataset_key == "fincausal" and RUN_MODE == "one_doc":
        if pre_gold_contract["gold_target_relation_count"] <= 0:
            raise RuntimeError(
                "FinCausal one-doc integrity guard stopped BEFORE API use: "
                "selected record has zero scored CAUSE relations."
            )
        if pre_gold_contract["gold_entity_count"] < 2:
            raise RuntimeError(
                "FinCausal one-doc integrity guard stopped BEFORE API use: "
                "selected record has fewer than two gold endpoints."
            )
    run_dir = RUNS_ROOT / dataset_key / RUN_MODE / safe_dir_name(rkey)
    run_dir.mkdir(parents=True, exist_ok=True)

    clean_record = expstate.strip_gold(gold_record)
    input_path = run_dir / "pipeline_input_NO_GOLD.jsonl"
    expstate.write_jsonl(input_path, [clean_record])
    assert not ({"entities","relations","pred_relations","ontology_links"} & set(clean_record))

    gold_path = run_dir / "POSTHOC_GOLD_AFTER_LAYER12.jsonl"
    if gold_path.exists():
        gold_path.unlink()

    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    if dataset_key == "eventstoryline":
        final_state = esl_v17.run_native_pipeline(
            project_root=PROJECT_ROOT, input_jsonl=input_path, ontology_path=ONTOLOGY_FILES[dataset_key],
            profile_path=cfg["profile"], guidance_path=cfg["guidance"], task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"], relation_aliases_path=cfg["aliases"], run_dir=run_dir,
            model_name=MODEL_NAME, api_key=api_key, host=OPENROUTER_HOST, workers=LAYER_WORKERS,
            max_tokens=MAX_TOKENS, request_timeout=REQUEST_TIMEOUT, reasoning_effort=REASONING_EFFORT,
            verbose=VERBOSE, clean_run_dir=False,
        )
        expstate.write_jsonl(gold_path, [{k:v for k,v in gold_record.items() if not k.startswith("__")}])
        summary = esl_v17.analyze_run(
            run_dir=run_dir, gold_jsonl=gold_path, catalog_path=cfg["catalog"], aliases_path=cfg["aliases"]
        )
        relation_metrics = summary.get("projected_relation_evaluation") or summary.get("strict_relation_evaluation") or {}
        endpoint_metrics = summary.get("relation_endpoint_evaluation") or summary.get("event_entity_evaluation") or {}
        result = {
            "dataset": dataset_key, "record_key": rkey, "document_id": gold_record.get("document_id"),
            "relation_metrics": relation_metrics, "endpoint_metrics": endpoint_metrics,
            "candidate_pool": summary.get("candidate_pool_coverage") or summary.get("candidate_pool") or {},
            "run_dir": str(run_dir),
        }
        adapters.write_json(run_dir / "posthoc_evaluation.json", result)
    else:
        final_state = adapters.run_native_pipeline_record(
            dataset_key=dataset_key, project_root=PROJECT_ROOT, input_jsonl=input_path,
            ontology_path=ONTOLOGY_FILES[dataset_key], profile_path=cfg["profile"], guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"], relation_catalog_path=cfg["catalog"], relation_aliases_path=cfg["aliases"],
            run_dir=run_dir, model_name=MODEL_NAME, api_key=api_key, host=OPENROUTER_HOST,
            workers=LAYER_WORKERS, max_tokens=MAX_TOKENS, request_timeout=REQUEST_TIMEOUT,
            reasoning_effort=REASONING_EFFORT, verbose=VERBOSE, clean_run_dir=False,
        )
        # Gold becomes available only AFTER native Layer 12 returned.
        expstate.write_jsonl(gold_path, [{k:v for k,v in gold_record.items() if not k.startswith("__")}])
        result = adapters.evaluate_state(dataset_key, final_state, gold_record)
        result.update({
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "run_dir": str(run_dir),
            "pre_run_gold_contract": pre_gold_contract,
        })

        # Post-L12 evaluator integrity check. If controller saw non-zero gold but the
        # evaluator sees zero, stop loudly instead of misclassifying the adapter.
        if dataset_key == "fincausal":
            evaluated_gold = int((result.get("relation_metrics") or {}).get("gold", 0) or 0)
            expected_gold = int(pre_gold_contract["gold_target_relation_count"])
            if expected_gold > 0 and evaluated_gold == 0:
                raise RuntimeError(
                    "FinCausal evaluation-integrity error AFTER Layer 12: "
                    f"controller saw {expected_gold} gold CAUSE relation(s), evaluator saw 0. "
                    "This is an evaluator/projection bug, not an extraction score."
                )
        adapters.write_json(run_dir / "posthoc_evaluation.json", result)

    result.setdefault("pre_run_gold_contract", pre_gold_contract)
    result["one_doc_sanity"] = one_doc_sanity(dataset_key, result)
    return result

print("Runner defined. No API call has been made by this cell.")

Runner defined. No API call has been made by this cell.


## Execute selected mode

In [8]:
all_results = []
all_failures = []

if not RUN_PAID:
    print("RUN_PAID=False -> STOPPED BEFORE ALL API CALLS. Preflight/audit only.")
    print("If the status table is correct, set RUN_PAID=True.")
else:
    for dataset_key in RUN_DATASETS:
        force = FORCE_RUN[dataset_key]
        entry = manifest[dataset_key]
        expstate.assert_run_allowed(manifest, dataset_key, RUN_MODE, force=force)

        if RUN_MODE == "one_doc" and entry.get("one_doc_completed") and not force:
            print(f"SKIP {dataset_key}: one-doc sanity already passed. NO API CALL.")
            continue

        if RUN_MODE == "smoke5" and not entry.get("one_doc_completed") and not force:
            print(f"SKIP {dataset_key}: one-doc sanity has NOT passed; smoke-5 is blocked.")
            continue

        selected = expstate.select_records_for_mode(
            dataset_key, dataset_rows[dataset_key], manifest, RUN_MODE,
            preferred_document_ids=SMOKE_DOCUMENT_IDS.get(dataset_key) or None,
        )
        expstate.save_manifest(LIVE_MANIFEST, manifest)

        if RUN_MODE == "smoke5":
            selected = expstate.pending_smoke_records(dataset_key, selected, manifest)
            if not selected:
                print(f"SKIP {dataset_key}: all fixed smoke-5 records already completed.")
                continue

        print(f"\n=== {dataset_key} | {RUN_MODE} | records to run now: {len(selected)} ===")
        if dataset_key == "fincausal" and RUN_MODE == "one_doc":
            contract = expstate.gold_contract_summary("fincausal", selected[0])
            print("FinCausal selected gold contract (controller only; NOT pipeline-visible):")
            pprint(contract)
            assert contract["gold_target_relation_count"] > 0
        for i, gold_record in enumerate(selected, 1):
            rkey = expstate.record_key(dataset_key, gold_record)
            print(f"[{i}/{len(selected)}] {rkey} | {gold_record.get('document_id')}")
            try:
                result = run_one_record(dataset_key, gold_record)
                all_results.append(result)

                if RUN_MODE == "one_doc":
                    sanity = result.get("one_doc_sanity") or one_doc_sanity(dataset_key, result)
                    manifest[dataset_key]["last_one_doc_result"] = {
                        "relation_metrics": result.get("relation_metrics"),
                        "endpoint_metrics": result.get("endpoint_metrics"),
                        "sanity": sanity,
                        "run_dir": result.get("run_dir"),
                    }
                    manifest[dataset_key]["best_version"] = CONFIGS[dataset_key]["version"]
                    if sanity["passed"]:
                        manifest[dataset_key]["one_doc_completed"] = True
                        manifest[dataset_key]["status"] = "READY_5"
                        manifest[dataset_key].pop("last_error", None)
                        print("  ONE-DOC SANITY: PASS -> READY_5")
                    else:
                        manifest[dataset_key]["one_doc_completed"] = False
                        manifest[dataset_key]["status"] = "NEEDS_FIX"
                        print("  ONE-DOC SANITY: FAIL -> NEEDS_FIX")
                else:
                    # The paid smoke-5 may only be executed once. Completion is recorded
                    # regardless of score; score determines later LOCK/acceptance, not reruns.
                    expstate.mark_record_complete(manifest, dataset_key, RUN_MODE, rkey)

                expstate.save_manifest(LIVE_MANIFEST, manifest)
                print("  relation:", result.get("relation_metrics"))
                print("  endpoint from FINAL RELATIONS:", result.get("endpoint_metrics"))
                if dataset_key == "maven_ere":
                    print("  Layer-1 inventory diagnostics:", result.get("inventory_endpoint_metrics"))
                    print("  MAVEN v1.7 relation-stage diagnostics:", result.get("maven_relation_stage_diagnostics"))
                    print("  MAVEN v1.7 post-L12 stage gold diagnostics:", result.get("maven_relation_stage_gold_diagnostics"))

            except Exception as exc:
                failure = {
                    "dataset": dataset_key, "record_key": rkey,
                    "document_id": gold_record.get("document_id"),
                    "error_type": type(exc).__name__, "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
                all_failures.append(failure)
                manifest[dataset_key]["status"] = "NEEDS_FIX" if RUN_MODE == "one_doc" else "SMOKE5_IN_PROGRESS"
                manifest[dataset_key]["last_error"] = failure["traceback"]
                expstate.save_manifest(LIVE_MANIFEST, manifest)
                print(f"FAILED {dataset_key}: {type(exc).__name__}: {exc}")
                print("Continuing so summaries are saved.")
                break

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    summary_path = RUNS_ROOT / f"summary_{RUN_MODE}_{timestamp}.json"
    failures_path = RUNS_ROOT / f"failures_{RUN_MODE}_{timestamp}.json"
    expstate.atomic_write_json(summary_path, all_results)
    expstate.atomic_write_json(failures_path, all_failures)
    print("\nSaved summary :", summary_path)
    print("Saved failures:", failures_path)
    print("Updated manifest:", LIVE_MANIFEST)


=== maven_ere | one_doc | records to run now: 1 ===
[1/1] maven_ere:0:e0ff956daea4cac8 | MAVEN_ERE - 2002c29914e6d8b5
[NeoOLAF] Run directory: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\unified4_v1_7\maven_ere\one_doc\maven_ere_0_e0ff956daea4cac8
[NeoOLAF] from_layer=0, to_layer=None, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.00s
[NeoOLAF] Layer 1/

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.04s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=4; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.14s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.01s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 43 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 64 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.01s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[Ne

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.02s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=4; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.01s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\unified4_v1_7\maven_ere\one_doc\maven_ere_0_e0ff956daea4cac8\exports
[NeoOLAF] Finished layer: layer12_serialization in 0.32s
[NeoOLAF] Pipeline finished in 65.02s
[NeoOLAF] Saved checkpoint: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\unified4_v1_7\maven_ere\one_doc\maven_ere_0_e0ff956daea4cac8\checkpoints\after_selected_pipeline.pkl.gz
[NeoOLAF] Total run time: 65.15s
  ONE-DOC SANITY: FAIL -> NEEDS_FIX
  rela

## Combined dashboard — recovered + current results

For FinCausal, the saved evaluation now carries `pre_run_gold_contract`, so an impossible `gold=0` projection cannot be mistaken for a real model score.


In [9]:
def compact_metric(m):
    if not isinstance(m, dict):
        return (None,None,None)
    return (m.get("precision"), m.get("recall"), m.get("f1"))

dashboard_results = {}

# Load the latest actual on-disk result for each non-ESL dataset, if present.
for k in ["fincausal","maven_ere","causalbank"]:
    r, p = newest_existing_one_doc_result(k)
    if r:
        dashboard_results[k] = {**r, "_source": str(p)}

# CausalBank v1.3 deterministic preview supersedes its old conservative relation score for development diagnostics.
if cb_preview.get("available"):
    dashboard_results["causalbank_preview_v1_3"] = {
        "dataset":"causalbank",
        "document_id":dataset_rows["causalbank"][0].get("document_id"),
        "relation_metrics":cb_preview.get("relation_metrics"),
        "endpoint_metrics":cb_preview.get("endpoint_metrics"),
        "_source":"ZERO-COST v1.3 deterministic dense projection preview",
    }

for r in all_results:
    dashboard_results[r["dataset"]] = {**r, "_source":"current invocation"}

rows=[]
for key,r in dashboard_results.items():
    ep=compact_metric(r.get("endpoint_metrics", {}))
    inv=(r.get("inventory_endpoint_metrics") or {})
    rel=compact_metric(r.get("relation_metrics", {}))
    rows.append({
        "row":key,
        "dataset":r.get("dataset"),
        "document_id":r.get("document_id"),
        "endpoint_P":ep[0],"endpoint_R":ep[1],"endpoint_F1":ep[2],
        "layer1_relation_endpoint_R":inv.get("relation_endpoint_cluster_recall"),
        "layer1_all_event_cluster_R":inv.get("all_gold_cluster_recall"),
        "relation_P":rel[0],"relation_R":rel[1],"relation_F1":rel[2],
        "source":r.get("_source") or r.get("run_dir"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    pprint(rows)

print("\nPersistent manifest:")
for k in expstate.DATASET_KEYS:
    e=manifest[k]
    print(k, "status=",e.get("status"),
          "version=",e.get("best_version"),
          "one_doc=",e.get("one_doc_completed"),
          "smoke5=",e.get("smoke5_already_run"),
          "locked=",e.get("locked"))

,row,dataset,document_id,endpoint_P,endpoint_R,endpoint_F1,layer1_relation_endpoint_R,layer1_all_event_cluster_R,relation_P,relation_R,relation_F1,source
0,fincausal,fincausal,FinCausal - a39155e8dec741b9,1.000000,1.000000,1.000000,NaN,NaN,1.000000,1.000000,1.000000,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...
1,maven_ere,maven_ere,MAVEN_ERE - 2002c29914e6d8b5,0.727273,0.666667,0.695652,1.0,1.0,0.000000,0.000000,0.000000,current invocation
2,causalbank,causalbank,CausalBank - e324949663db4295,1.000000,0.727273,0.842105,NaN,NaN,1.000000,0.145631,0.254237,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...
3,causalbank_preview_v1_3,causalbank,CausalBank - e324949663db4295,1.000000,1.000000,1.000000,NaN,NaN,0.936364,1.000000,0.967136,ZERO-COST v1.3 deterministic dense projection ...



Persistent manifest:
eventstoryline status= READY_5 version= v1.7 one_doc= True smoke5= False locked= False
fincausal status= READY_5 version= unified-v1.3.1-selection-hotfix one_doc= True smoke5= False locked= False
maven_ere status= NEEDS_FIX version= unified-v1.7 one_doc= False smoke5= False locked= False
causalbank status= READY_5 version= unified-v1.3 one_doc= True smoke5= False locked= False


## Run order for MAVEN v1.7

1. Run the entire notebook with `RUN_PAID=False`.
2. Confirm ontology/anti-leak preflight is green and inspect the **MAVEN frozen-v1.4 ZERO-COST source rescue ceiling**. It should remain at 100% on the frozen development document. v1.7 changes relation reasoning only; the v1.4 event inventory remains frozen.
3. Confirm EventStoryLine, FinCausal and CausalBank remain `READY_5`; MAVEN should be the only `NEEDS_FIX` dataset.
4. Set only `RUN_PAID=True`. The notebook defaults to `RUN_DATASETS=["maven_ere"]`, so only the frozen MAVEN one-doc development record runs.
5. After Layer 12, inspect relation metrics, `inventory_endpoint_metrics`, `maven_relation_stage_diagnostics`, and `maven_relation_stage_gold_diagnostics`. The latter is computed only after Layer 12 and shows candidate/verifier/final gold-pair recall without exposing gold to the pipeline.
6. Do not run smoke-5 until this one-doc configuration is accepted.


In [10]:
# Example only — leave commented until intentionally freezing after the ONE allowed smoke-5.
#
# dataset_to_lock = "fincausal"
# assert manifest[dataset_to_lock].get("smoke5_already_run"), "Do not lock before the single smoke-5 is complete."
# manifest[dataset_to_lock]["locked"] = True
# manifest[dataset_to_lock]["status"] = "LOCKED"
# expstate.save_manifest(LIVE_MANIFEST, manifest)
# print(dataset_to_lock, "LOCKED")